In [27]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import log_loss, roc_auc_score, f1_score
from itertools import product

In [28]:
from sklearn.metrics import (roc_auc_score, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score, f1_score)

# Global namefiles used for read csv files

In [29]:
global_path_train = 'data/final_train_data_transmission.csv'
global_path_test  = 'data/final_test_data_transmission.csv'
global_path_val   = 'data/final_val_data_transmission.csv'

In [30]:
# 1. Cargar datasets (asegurando el nombre correcto de las columnas)
df_train = pd.read_csv(global_path_train)
df_test = pd.read_csv(global_path_test)
df_val = pd.read_csv(global_path_val)

if ' minPowerdBm' in df_train.columns:
    df_train.rename(columns={' minPowerdBm': 'minPowerdBm'}, inplace=True)
    df_test.rename(columns={' minPowerdBm': 'minPowerdBm'}, inplace=True)
    df_val.rename(columns={' minPowerdBm': 'minPowerdBm'}, inplace=True)


In [20]:
df_train.head()

df_train[df_train['id_simulation'] == 'sim_alt_600_raan_200']['srcId'].value_counts()

srcId
4     10
42     9
7      9
47     9
51     9
11     9
15     9
14     9
5      9
9      9
27     9
13     9
24     9
6      9
31     9
38     9
59     9
2      9
43     9
46     9
49     9
57     9
50     9
45     9
34     9
41     9
33     9
20     9
28     9
40     9
55     9
17     9
12     9
58     9
54     9
21     9
16     9
53     9
25     9
26     9
44     9
18     9
1      9
48     9
39     9
32     8
19     8
10     8
52     8
22     8
37     8
3      8
56     8
0      8
36     8
8      8
30     8
23     8
29     8
35     8
Name: count, dtype: int64

In [24]:
aux_df.head()

0    42
1     7
2    47
3    32
4    51
Name: srcId, dtype: int64

In [26]:
aux_df = df_train[df_train['id_simulation'] == 'sim_alt_600_raan_200']

aux_df[aux_df['srcId'] == 42]

,pid,time,srcId,dstId,srcSat,dstSat,latDev,longDev,loraTP,loraCF,...,satId,elevSat,doppler,rcvOk,minPowerdBm,sensitivitydBm,duration,alt,raan,id_simulation
0,31430,11.1269,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,4.47837,19929.80,0,-134.951,-137,1.58106,600,200,sim_alt_600_raan_200
51,32031,101.6120,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,12.38780,19104.30,0,-132.458,-137,1.58106,600,200,sim_alt_600_raan_200
115,32748,205.8620,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,27.22610,16056.20,1,-128.700,-137,1.58106,600,200,sim_alt_600_raan_200
174,33431,304.5100,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,49.27580,4887.99,1,-125.219,-137,1.58106,600,200,sim_alt_600_raan_200
236,34132,405.1710,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,37.40600,-12225.60,1,-126.800,-137,1.58106,600,200,sim_alt_600_raan_200
297,34842,509.2330,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,17.80170,-18103.90,1,-130.884,-137,1.58106,600,200,sim_alt_600_raan_200
388,36607,977.3910,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,-13.42410,-19614.40,0,-140.661,-137,1.58106,600,200,sim_alt_600_raan_200
448,37269,1070.2600,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,-17.28640,-19196.20,0,-141.740,-137,1.58106,600,200,sim_alt_600_raan_200
505,37909,1167.8400,42,-1,-1,-1,57.8977,24.2191,0.015849,868000000.0,...,0,-21.13370,-18732.20,0,-142.727,-137,1.58106,600,200,sim_alt_600_raan_200


In [31]:
def grid_search(df_val, grid_params):
    print('grid_params: ', grid_params)
    #### Iterar sobre time e id_simulation
    ###df_train = df_train.sort_values('time').reset_index(drop=True)
    
    BW = 125000               
    base_p_link = df_val['minPowerdBm'] - df_val['sensitivitydBm']
        
    BW_SF = BW / (2 ** df_val['loraSF'])
    base_p_doppler = np.abs(df_val['doppler']) / BW_SF
        
    ToA = df_val['duration']
    ###times = df_train['time'].values
    y_true = df_val['rcvOk'].values
    
    best_loss = float('inf')
    best_params = None
    
    # Sort by id_simulaiton and time    
    df_val_aux = df_val.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
    
    #print('before first for')
    for T_v in grid_params['T_window']:
        ##############################################################################
        ###############################################################################
        ##############################################################################     
        
        lambda_trx_list = []
                
        for sim_id, group in df_val_aux.groupby('id_simulation'):
            times = group['time'].values
            # Get limits (left, right) of each row accordding to the T_v/2
            lim_left = times - (T_v / 2)
            lim_right = times + (T_v / 2)

            # Get idxs of the previous calculated limits 
            idx_left = np.searchsorted(times, lim_left, side='left')
            idx_right = np.searchsorted(times, lim_right, side='right')

            # Calculate the number of trxs/registers between the idxs
            num_trx_window = np.maximum((idx_right - idx_left) - 1, 0)        
            lambda_trx = num_trx_window / T_v
            lambda_trx_list.append(lambda_trx)




        # Unimos las densidades calculadas (mantienen el mismo orden que el dataframe)
        # Concatenate traffic density , they kep the original order after the sort by id_simulation, time
        lambda_traffic = np.concatenate(lambda_trx_list)


        ##############################################################################
        ##############################################################################
        ##############################################################################
        P_collision = np.exp(-2 * lambda_traffic * ToA)
        
        #print('before second for')
        for alpha, k in product(grid_params['alpha'], grid_params['k']):            
            # Calculate p_doppler
            P_doppler = np.exp(-alpha * base_p_doppler)
            
            # Calculate p_link
            P_link = 1 / (1 + np.exp(-k * base_p_link))
            
            # Final probability
            y_pred = P_link * P_doppler * P_collision
            y_pred_clipped = np.clip(y_pred, 1e-15, 1 - 1e-15)
            
            loss = log_loss(y_true, y_pred_clipped)
            ''' 
            print('*'*50)            
            print(f"-> T_v: {T_v:.4f}")
            print(f"-> alpha: {alpha:.4f}")
            print(f"-> k: {k:.4f}")
            print('loss: ', loss)
            '''
            if loss < best_loss:
                best_loss = loss
                best_params = {'T_window': T_v, 'alpha': alpha, 'k': k}

    return best_params, best_loss

In [32]:
def evaluate_test(df_test, params):
    #### Iterar sobre time e id_simulation
    #####df_test = df_test.sort_values('time').reset_index(drop=True)
    # Sort by id_simulaiton and time    
    df_test_aux = df_test.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

    BW = 125000                           
    BW_SF = BW / (2 ** df_test['loraSF'])
    
    # P_link    
    base_p_link = df_test_aux['minPowerdBm'] - df_test_aux['sensitivitydBm']
    P_link = 1 / (1 + np.exp(-params['k'] * base_p_link))
    
    # P_doppler
    base_p_doppler = np.abs(df_test_aux['doppler']) / BW_SF
    P_doppler = np.exp(-params['alpha'] * base_p_doppler)
    
    # P_collision    
    T_v = params['T_window']
    ToA = df_test_aux['duration']
    lambda_trx_list = []
    
    # Iteramos sobre cada simulación para que no haya "colisiones fantasma" entre ellas
    for sim_id, group in df_test_aux.groupby('id_simulation'):
        times = group['time'].values
        # Get limits (left, right) of each row accordding to the T_v/2
        lim_left = times - (T_v / 2)
        lim_right = times + (T_v / 2)

        # Get idxs of the previous calculated limits 
        idx_left = np.searchsorted(times, lim_left, side='left')
        idx_right = np.searchsorted(times, lim_right, side='right')

        # Calculate the number of trxs/registers between the idxs
        num_trx_window = np.maximum((idx_right - idx_left) - 1, 0)        
        lambda_trx = num_trx_window / T_v
        lambda_trx_list.append(lambda_trx)
    
    # Concatenate traffic density , they kep the original order after the sort by id_simulation, time
    lambda_traffic = np.concatenate(lambda_trx_list)
    P_collision = np.exp(-2 * lambda_traffic * ToA)    
    
    # Predicción final
    df_test_aux['P_RcvOk_Pred'] = P_link * P_doppler * P_collision
    
    y_true = df_test_aux['rcvOk'].values
    y_pred = df_test_aux['P_RcvOk_Pred'].values
    
    loss = log_loss(y_true, np.clip(y_pred, 1e-15, 1 - 1e-15))
    auc = roc_auc_score(y_true, y_pred)
    f1 = f1_score(y_true, (y_pred > 0.5).astype(int))
    
    return loss, auc, f1, y_true, y_pred

In [33]:
# =====================================================
# GRID SEARCH
# ======================================================
grid_params = {
    'T_window': [1, 3, 5, 10, 20],
    'k': [0.5, 1, 2, 3, 4, 5],    
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1] 
}

print("GridSearch...")
best_params, min_loss_train = grid_search(df_val, grid_params)
print(f"-> Best params: {best_params}")

print("Testing...")
loss_test, auc_test, f1_test, y_true, y_pred = evaluate_test(df_test, best_params)

threshold = 0.5  # Puedes cambiar esto por el umbral óptimo calibrado si lo deseas
y_pred_bin = (y_pred > threshold).astype(int)

acc_cal = accuracy_score(y_true, y_pred_bin)
prec_cal = precision_score(y_true, y_pred_bin)
rec_cal = recall_score(y_true, y_pred_bin)
f1_cal = f1_score(y_true, y_pred_bin)

print(f"-> Log-Loss: {loss_test:.4f}")
print(f"-> AUC-ROC:  {auc_test:.4f}")
print(f"-> F1-Score: {f1_test:.4f}")
print(f"-> Accuracy:  {acc_cal:.4f}")
print(f"-> Precision: {prec_cal:.4f}")
print(f"-> Recall:    {rec_cal:.4f}")
print(f"-> F1-Score:  {f1_cal:.4f}")

GridSearch...
grid_params:  {'T_window': [1, 3, 5, 10, 20], 'k': [0.5, 1, 2, 3, 4, 5], 'alpha': [0.0001, 0.001, 0.01, 0.1, 1]}
-> Best params: {'T_window': 3, 'alpha': 0.0001, 'k': 5}
Testing...
-> Log-Loss: 0.5045
-> AUC-ROC:  0.9382
-> F1-Score: 0.6375
-> Accuracy:  0.7476
-> Precision: 0.9375
-> Recall:    0.4830
-> F1-Score:  0.6375


In [34]:
# Find the threshold
precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred)

# Calculamos el F1-Score para cada posible umbral probado por la función
# Sumamos 1e-10 en el denominador para evitar divisiones matemáticas por cero
# Find the F1-Score for each threshold 
f1_scores_by_threshold = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

# Extraemos el umbral que dio el F1-Score más alto
best_idx = np.argmax(f1_scores_by_threshold)
best_threshold = thresholds[best_idx]

In [35]:



print(f"=== CALIBRACIÓN DE UMBRAL ===")
# Comparativa con el umbral "ingenuo" del 50%
#f1_ingenuo = f1_score(y_true, (y_pred > 0.5).astype(int))
f1_dummy = f1_score(y_true, (y_pred > 0.5).astype(int))

print(f"F1-Score with threshold (0.50): {f1_dummy:.4f}")
print(f"Best threshold: {best_threshold:.6f}\n")

# 6. Evaluar el clasificador final aplicando el nuevo umbral óptimo
# Evaluate with the correct threshold
y_pred_fit = (y_pred > best_threshold).astype(int)

acc_cal = accuracy_score(y_true, y_pred_fit)
prec_cal = precision_score(y_true, y_pred_fit)
rec_cal = recall_score(y_true, y_pred_fit)
f1_cal = f1_score(y_true, y_pred_fit)

print(f"=== MÉTRICAS FINALES CON UMBRAL ÓPTIMO ({best_threshold:.6f}) ===")
print(f"Accuracy:  {acc_cal:.4f}")
print(f"Precision: {prec_cal:.4f}")
print(f"Recall:    {rec_cal:.4f}")
print(f"F1-Score:  {f1_cal:.4f}")

=== CALIBRACIÓN DE UMBRAL ===
F1-Score with threshold (0.50): 0.6375
Best threshold: 0.114350

=== MÉTRICAS FINALES CON UMBRAL ÓPTIMO (0.114350) ===
Accuracy:  0.8712
Precision: 0.8177
Recall:    0.9261
F1-Score:  0.8686


In [ ]:
best_params

In [ ]:
for alpha, k in product(grid_params['alpha'], grid_params['k']):       
    print(alpha, k)
    #print('alpha: ', alpha)
    #print('k: ', k)

In [13]:
df_val.head()

,pid,time,srcId,dstId,srcSat,dstSat,latDev,longDev,loraTP,loraCF,...,satId,elevSat,doppler,rcvOk,minPowerdBm,sensitivitydBm,duration,alt,raan,id_simulation
0,42198,450.861,49,-1,-1,-1,45.3580,8.43519,0.025119,868000000.0,...,0,35.9168,11301.60,0,-126.615,-135,0.790528,740,160,sim_alt_740_raan_160
1,42326,451.686,38,-1,-1,-1,44.8130,14.05660,0.019953,868000000.0,...,0,26.6111,7519.62,1,-129.291,-133,0.436224,740,160,sim_alt_740_raan_160
2,42334,453.326,30,-1,-1,-1,49.7013,18.48320,0.019953,868000000.0,...,0,27.4724,-2423.88,1,-129.120,-135,0.790528,740,160,sim_alt_740_raan_160
3,42347,455.384,2,-1,-1,-1,52.0370,15.38900,0.025119,868000000.0,...,0,36.0161,-5792.43,1,-126.600,-133,0.436224,740,160,sim_alt_740_raan_160
4,42350,455.764,56,-1,-1,-1,37.2872,14.03830,0.015849,868000000.0,...,0,12.6457,12845.70,1,-133.596,-137,1.581060,740,160,sim_alt_740_raan_160


In [14]:
df_test.head()

,pid,time,srcId,dstId,srcSat,dstSat,latDev,longDev,loraTP,loraCF,...,satId,elevSat,doppler,rcvOk,minPowerdBm,sensitivitydBm,duration,alt,raan,id_simulation
0,31499,11.1269,42,-1,-1,-1,57.8977,24.21910,0.015849,868000000.0,...,0,5.46828,18219.6,1,-134.626,-137,1.581060,600,190,sim_alt_600_raan_190
1,31506,12.3645,7,-1,-1,-1,48.8319,14.18630,0.025119,868000000.0,...,0,-4.39718,19642.5,0,-135.901,-137,1.581060,600,190,sim_alt_600_raan_190
2,31513,13.0547,47,-1,-1,-1,59.7683,8.16129,0.025119,868000000.0,...,0,6.01024,20185.9,0,-132.451,-135,0.790528,600,190,sim_alt_600_raan_190
3,31515,13.3005,32,-1,-1,-1,50.5868,13.24900,0.025119,868000000.0,...,0,-2.86111,19800.3,0,-135.394,-135,0.790528,600,190,sim_alt_600_raan_190
4,31522,15.3593,51,-1,-1,-1,42.3684,27.90230,0.019953,868000000.0,...,0,-8.80175,16756.1,0,-138.314,-137,1.581060,600,190,sim_alt_600_raan_190
